### import + path

In [3]:
import pandas as pd
from pathlib import Path
import sqlite3

### convert to sqlite

In [15]:
#street_crime col

import sqlite3
import pandas as pd
from pathlib import Path

#paths/connection
path = Path("data")
db_name = "data/police_data.db"
db_connection = sqlite3.connect(db_name)

#files end in 3 ways: *-street, *-stop-and-search, *-outcomes, edit to work with new
orig_files = list(path.rglob("*-street.csv"))
print(len(orig_files))

#renaming/defining which ones to keep
#dropped: lsoa name
column_mapping = {
    'Crime ID': 'crime_id',
    'Month': 'month',
    'Reported by': 'reported_by',
    'Falls within': 'falls_within',
    'Longitude': 'longitude',
    'Latitude': 'latitude',
    'Location': 'location',
    'LSOA code': 'lsoa_code',
    'Crime type': 'crime_type',
    'Last outcome category': 'last_outcome',
    'Context': 'context'
}

#start count
successful = 0

for i, file in enumerate(orig_files):
    try:
        #takes only columns mentioned before, low_memory bc pandas warnings
        df = pd.read_csv(file, usecols=column_mapping.keys(), low_memory=False)
            
        #rename the columns as above
        df = df.rename(columns=column_mapping)
        
        #will convert to date timesetamp from b/c no other way to make it nice to filter afaik
        df['month'] = pd.to_datetime(df['month'])
            
        #write to db
        df.to_sql("street_crimes", db_connection, if_exists="append", index=False)
        successful += 1
        
        if (i + 1) % 50 == 0:
            print(f"processed {i + 1} files")
                
    except Exception as e:
        print(f"not processed: {file.name}: {e}")



#index for based on lsoa
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_lsoa ON street_crimes(lsoa_code);")

#index for crimes by date
db_connection.execute("CREATE INDEX IF NOT EXISTS idx_street_month ON street_crimes(month);")


db_connection.close()
print(f"{successful} files into {db_name}")

8107
processed 50 files
processed 100 files
processed 150 files
processed 200 files
processed 250 files
processed 300 files
processed 350 files
processed 400 files
processed 450 files
processed 500 files
processed 550 files
processed 600 files
processed 650 files
processed 700 files
processed 750 files
processed 800 files
processed 850 files
processed 900 files
processed 950 files
processed 1000 files
processed 1050 files
processed 1100 files
processed 1150 files
processed 1200 files
processed 1250 files
processed 1300 files
processed 1350 files
processed 1400 files
processed 1450 files
processed 1500 files
processed 1550 files
processed 1600 files
processed 1650 files
processed 1700 files
processed 1750 files
processed 1800 files
processed 1850 files
processed 1900 files
processed 1950 files
processed 2000 files
processed 2050 files
processed 2100 files
processed 2150 files
processed 2200 files
processed 2250 files
processed 2300 files
processed 2350 files
processed 2400 files
process

In [19]:
##create lsoa_demographic

file_path = "data/2025_all_iod_scores_ranks_deciles.csv" 
db_name = "data/police_data.db"
db_connection = sqlite3.connect(db_name)

#define new column for easiness later
column_mapping = {
    'LSOA code': 'lsoa_code',
    'Income Score': 'income_score',
    'Employment Score': 'employment_score',
    'Education Skills and Training Score': 'education_score',
    'Health Deprivation and Disability Score': 'health_score',
    'Barriers to Housing and Services Score': 'barrier_score',
    'Living Environment Score': 'living_score',
    'Total population': 'pop',
    'Dependent Children': 'child_pop',
    'Older population': 'old_pop',
    'Working age population': 'working_pop'
}

try:
    #load only columns mentioned in column_mapping
    df = pd.read_csv(file_path, usecols=column_mapping.keys())
    
    #rename the columns
    df = df.rename(columns=column_mapping)
    
    #write to db
    df.to_sql("lsoa_demographics", db_connection, if_exists="replace", index=False)
    print(len(df))
    
    #create index for faster querying
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_demographics(lsoa_code);")

#just in case
except Exception as e:
    print(f"not read: {e}")


db_connection.close()

33755


In [21]:
##create lsoa_info

file_path = "data/2025_all_iod_scores_ranks_deciles.csv" 
db_name = "data/police_data.db"
db_connection = sqlite3.connect(db_name)

#define new column for easiness later
column_mapping = {
    'LSOA code': 'lsoa_code',
    'LSOA name': 'lsoa_name',
    'Local Authority District code': 'loc_auth_code',
    'Local Authority District name': 'loc_auth_name'
}

try:
    #load only columns mentioned in column_mapping
    df = pd.read_csv(file_path, usecols=column_mapping.keys())
    
    #rename the columns
    df = df.rename(columns=column_mapping)
    
    #write to db
    df.to_sql("lsoa_info", db_connection, if_exists="replace", index=False)
    print(len(df))
    
    #create index for faster querying
    db_connection.execute("CREATE INDEX IF NOT EXISTS idx_lsoa_code ON lsoa_info(lsoa_code);")

#just in case
except Exception as e:
    print(f"not read: {e}")


db_connection.close()
print(f"merged {successful} files into '{db_name}'.")

33755
merged 8107 files into 'data/police_data.db'.
